# Step 3: corrupted images

Runs the 15 MedMNIST-C corruptions at 5 severities over the val + test images and saves what each model outputs. Uses the same Drive folder and `GH_TOKEN` as notebooks 1 and 2.

1. Runtime > Change runtime type > T4 GPU.
2. Run the cells in order and keep the tab open.
3. If Colab disconnects, run everything again. Finished corruptions get skipped.

After any git push, reset the runtime first or this keeps running the old code.

In [ ]:
import os
import torch
from google.colab import drive, userdata

assert torch.cuda.is_available(), "no gpu, change the runtime type to T4"
print(torch.cuda.get_device_name(0))

drive.mount("/content/drive")

token = userdata.get("GH_TOKEN")
!git clone -q https://{token}@github.com/adipatel11/gr-sats-term-paper.git /content/repo || git -C /content/repo pull -q
%cd /content/repo/code
!pip install -q -r requirements.txt

# medmnist-c pins an old scikit-image that won't build here, so skip its dependencies.
# its motion blur needs imagemagick, and apt needs an update first or it 404s
!apt-get -qq update
!apt-get -qq install -y libmagickwand-dev > /dev/null
!pip install -q --no-deps medmnistc

# outputs go to drive so they survive a disconnect
os.environ["GRSATS_OUT"] = "/content/drive/MyDrive/grsats/outputs"

In [ ]:
# unzip to the colab disk, reading from drive is really slow
ham = "../data/HAM10000/dataverse_files"
!mkdir -p {ham}
!unzip -qn /content/drive/MyDrive/grsats/HAM10000_images_part_1.zip -d {ham}/HAM10000_images_part_1
!unzip -qn /content/drive/MyDrive/grsats/HAM10000_images_part_2.zip -d {ham}/HAM10000_images_part_2
!unzip -qn /content/drive/MyDrive/grsats/finalfitz17k.zip -d ../data

# counts should match the paper and missing image files should be 0
!python check_data.py
!python -m pytest -q

In [ ]:
# refit the probes so they get saved too (cpu, a few minutes). same features and
# settings as notebook 1, so the results should come out the same as before
for ds in ["ham", "fitz"]:
    for bb in ["dinov2", "biomedclip"]:
        !python train_probes.py --dataset {ds} --backbone {bb}

In [ ]:
# quick test on 64 images: clean check, time per corruption, and a picture to look at
from IPython.display import Image
!python corrupt_extract.py --dataset ham --models frozen --test
Image(os.environ["GRSATS_OUT"] + "/corruption_preview.png")

In [ ]:
# the pilot: ham, frozen backbones (the probes read these features)
!python corrupt_extract.py --dataset ham --models frozen

## After the pilot

Only run these once the pilot says GR-SATS is worth it. They take longer.

In [ ]:
!python corrupt_extract.py --dataset ham --models lora
!python corrupt_extract.py --dataset fitz --models frozen
!python corrupt_extract.py --dataset fitz --models lora